In [1]:
from analytics.db import query

In [2]:
query("""---What percent of all reviews give 5 stars? Round to 1 decimal.
SELECT
    ROUND(100.0 * COUNT(*) FILTER (WHERE review_score = 5) / COUNT(*), 1) AS pct_5_star_reviews
FROM order_reviews;""")

,pct_5_star_reviews
0,57.8


> **Interpretation:** Calculates the overall proportion of 5-star reviews across all customer feedback, showing high satisfaction prevalence.

In [3]:
query("""
      --way1
    SELECT 
        ROUND(SUM(i.price),2) AS way_1_items_revenue
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
""")

,way_1_items_revenue
0,854686.33


**Interpretation (Way 1):** Computes total revenue strictly from item prices for orders placed in August 2018 across all order statuses, without excluding canceled or unavailable orders[cite: 1].

In [4]:
query("""
      --way2
    SELECT 
        ROUND(SUM(p.payment_value),2) AS way_2_payments_revenue
    FROM order_payments AS p
    JOIN orders AS o ON o.order_id = p.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
""")

,way_2_payments_revenue
0,1022425.32


> **Interpretation (Way 2):** Computes total revenue based on payment values registered for August 2018 orders.

In [5]:
query("""
      --way3
    SELECT 
        p.payment_type,
        SUM(i.price) AS revenue
    FROM order_items AS i
    JOIN order_payments AS p ON p.order_id = i.order_id
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY p.payment_type
    ORDER BY revenue DESC
""")

,payment_type,revenue
0,credit_card,695780.67
1,boleto,118214.12
2,debit_card,39780.61
3,voucher,30882.00


> **Interpretation (Way 3):** Shows revenue broken down by payment type using a direct join, which inflates figures due to multi-payment duplication.

In [6]:
query("""
    SELECT
        o.order_status,
        p.payment_type,
        COUNT(DISTINCT o.order_id) AS orders_count
    FROM orders AS o
    JOIN order_payments AS p ON p.order_id = o.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND NOT EXISTS (
          SELECT 1 FROM order_items AS i WHERE i.order_id = o.order_id
      )
    GROUP BY o.order_status, p.payment_type
    ORDER BY orders_count DESC
""")

,order_status,payment_type,orders_count
0,canceled,voucher,51
1,unavailable,credit_card,5
2,canceled,not_defined,2
3,unavailable,boleto,1
4,unavailable,debit_card,1


Out of the 60 orders with payments but no items in August 2018, the vast majority are canceled or unavailable orders, with credit card being the most common payment method."

In [7]:
query("""
    SELECT 
        SUM(i.price) AS way_3_total_revenue
    FROM order_items AS i
    JOIN order_payments AS p ON p.order_id = i.order_id
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
""")

,way_3_total_revenue
0,884657.4


> **Interpretation:** Summarizes the flawed total revenue resulting from the multi-payment item duplication join.

In [8]:
august_reconciliation_query = """
    WITH item_agg AS (
        -- Group item prices and freight per order
        SELECT order_id, SUM(price) AS total_price, SUM(freight_value) AS total_freight
        FROM order_items
        GROUP BY order_id
    ),
    payment_agg AS (
        -- Group payments and count payment rows per order
        SELECT order_id, SUM(payment_value) AS total_payment, COUNT(*) AS payment_rows
        FROM order_payments
        GROUP BY order_id
    )
    -- Combine with August 2018 orders using LEFT JOIN
    SELECT 
        o.order_id,
        o.order_status,
        i.total_price,
        i.total_freight,
        p.total_payment,
        p.payment_rows
    FROM orders AS o
    LEFT JOIN item_agg AS i ON i.order_id = o.order_id
    LEFT JOIN payment_agg AS p ON p.order_id = o.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
"""

> **Interpretation:** Defines the base CTE for August 2018 orders to bridge items, freight, and payments.

In [9]:
query(f"""
    WITH base AS ({august_reconciliation_query})
    SELECT
        SUM(total_price) AS way_1_items,
        SUM(total_freight) AS total_freight,
        SUM(CASE WHEN total_price IS NULL THEN total_payment ELSE 0 END) AS payments_with_no_items,
        SUM(total_payment - total_price - total_freight) AS small_differences,
        SUM(total_payment) AS way_2_payments
    FROM base
""")

,way_1_items,total_freight,payments_with_no_items,small_differences,way_2_payments
0,854686.33,148622.14,19194.21,-77.36,1022425.32


> **Interpretation:** Successfully reconciles the gap between Way 1 (Items) and Way 2 (Payments) by accounting for freight and unlinked payments.

In [10]:
query("""
    WITH multi_pay AS (
        -- Find orders with more than one payment row
        SELECT order_id
        FROM order_payments
        GROUP BY order_id
        HAVING COUNT(*) > 1
    )
    SELECT 
        SUM(i.price) AS duplicated_item_prices
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND i.order_id IN (SELECT order_id FROM multi_pay)
""")

,duplicated_item_prices
0,23229.25


**Interpretation:** Represents an initial, partial estimate (23,229.25) of the extra revenue from multi-payment orders. This value underestimates orders with 3 or more payment rows, as the correct bridge extra is 29,971.07.

In [11]:
# 1. Confirmed Primary Figure (Products sold, excluding cancelled/unavailable items)
query("""
    SELECT 
        SUM(i.price) AS official_revenue
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND o.order_status NOT IN ('canceled', 'unavailable')
""")

,official_revenue
0,848860.1


> **Interpretation:** Derives Olist's official net revenue for August 2018 by excluding canceled and unavailable orders.

In [12]:
# 2. Sensitivity of Options: Calculating the figure if shipping costs are included
query("""
    SELECT 
        SUM(i.price + i.freight_value) AS revenue_with_freight
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND o.order_status NOT IN ('canceled', 'unavailable')
""")

,revenue_with_freight
0,996973.51


> **Interpretation:** Evaluates revenue sensitivity when shipping freight costs are added to item prices.

In [13]:
# 3. Sensitivity of the options: Calculating the figure if we consider only delivered orders.
query("""
    SELECT 
        SUM(i.price) AS revenue_delivered_only
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND o.order_status = 'delivered'
""")

,revenue_delivered_only
0,838576.64


> **Interpretation:** Checks revenue sensitivity when restricting orders exclusively to successfully delivered status.

In [14]:
query(f"""
    WITH base AS ({august_reconciliation_query})
    SELECT
        COUNT(*) FILTER (WHERE payment_rows > 1) AS orders_counted_again,
        SUM(total_price * (payment_rows - 1)) AS join_extra
    FROM base
    WHERE total_price IS NOT NULL
""")

,orders_counted_again,join_extra
0,139,29971.07


> **Interpretation:** Measures the exact row multiplier effect and extra monetary inflation from multi-payment joins.

In [15]:
query(f"""
    WITH base AS ({august_reconciliation_query}),
    installments AS (
        SELECT order_id, MAX(payment_installments) AS payment_installments
        FROM order_payments
        GROUP BY order_id
    )
    SELECT 
        b.order_id,
        b.total_price,
        b.total_freight,
        b.total_payment,
        (b.total_payment - b.total_price - b.total_freight) AS diff,
        i.payment_installments
    FROM base AS b
    JOIN installments AS i ON i.order_id = b.order_id
    WHERE b.total_price IS NOT NULL
      AND ABS(b.total_payment - b.total_price - b.total_freight) > 0 
      AND ABS(b.total_payment - b.total_price - b.total_freight) < 5
""")

,order_id,total_price,total_freight,total_payment,diff,payment_installments
0,602fd0f12980f5a92b67acc19187fb87,132.82,20.00,152.81,-0.01,15
1,34ae480fe2cec3fa921d33b72b32572a,570.00,52.98,622.99,0.01,1
2,b38b3526b8b8fdc807e8a0a42ab78573,22.32,7.74,30.19,0.13,2
3,9d4f40b767fac576ddb5e71af7db166c,150.00,20.00,169.99,-0.01,10
4,79d316a7d0d628895124bcefd1040ed7,479.80,19.00,498.79,-0.01,1
5,1cd147d1c0fe18f3b742a353396c44a7,84.00,20.00,103.99,-0.01,6
6,4d680edbaa7d3d9bed69532957368a03,286.98,66.10,353.09,0.01,10
7,0ec20feeac30045e916578f17c2085e2,411.00,39.00,449.99,-0.01,1
8,6fa0c125ee7d870f6602c97e33d87bc5,31.99,15.32,45.71,-1.60,1
9,051fcda88d997d3ff86012da2a556342,49.00,7.60,51.70,-4.90,4


> **Interpretation (5.2):** Orders with small differences predominantly involve single-installment payments where minor rounding or freight adjustments account for the cents-level variances.